Try our best to map PR GitHub username to country.

Would try:
1. `geopy` to normalize free-form "location" in GitHub profile page;
2. If NA, try https://pymatgen.org/team.html

In [2]:
# Get PR info: username, location and number of PRs from GitHub

!uv pip install requests

import csv
import os
import time

import requests

GITHUB_TOKEN: str = os.getenv("GITHUB_TOKEN", "")

HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"token {GITHUB_TOKEN}",
}


def get_all_pr_authors_with_counts(repo: str) -> dict[str, int]:
    print("Fetching merged PR authors (excluding bots)...")
    author_counts: dict[str, int] = {}
    page: int = 1
    per_page: int = 100

    while True:
        url = f"https://api.github.com/repos/{repo}/pulls"
        params = {"state": "closed", "per_page": per_page, "page": page}
        response = requests.get(url, headers=HEADERS, params=params)

        if response.status_code != 200:
            print(f"Failed to fetch PRs: {response.status_code}, {response.text}")
            break

        prs = response.json()
        if not prs:
            break

        for pr in prs:
            if not pr.get("merged_at"):
                continue

            user = pr.get("user")
            username = user.get("login", "") if user else ""
            if not username or not user:
                raise ValueError(f"Cannot get username for {user}")

            if username and "[bot]" not in username:
                author_counts[username] = author_counts.get(username, 0) + 1

        print(
            f"Fetched page {page}, total unique merged authors so far: {len(author_counts)}"
        )
        page += 1

    return author_counts


def get_user_location(username: str) -> str | None:
    url = f"https://api.github.com/users/{username}"
    response = requests.get(url, headers=HEADERS)
    if response.status_code == 200:
        data = response.json()
        return data.get("location")
    else:
        print(f"Failed to fetch user {username}: {response.status_code}")
        return None


repo: str = "materialsproject/pymatgen"
author_counts = get_all_pr_authors_with_counts(repo)

print("\nFetching PR info...")
user_locations: list[dict[str, str | int | None]] = []
for i, (username, count) in enumerate(sorted(author_counts.items())):
    location: str | None = get_user_location(username)
    user_locations.append(
        {"username": username, "location": location, "pr_count": count}
    )
    print(f"{i + 1:3d}/{len(author_counts)} {username}: {location} ({count} PRs)")
    time.sleep(1)

# Save to CSV
with open("pr_info.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["username", "location", "pr_count"])
    writer.writeheader()
    writer.writerows(user_locations)

print("Saved pr_info.csv")

Using Python 3.14.3 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Resolved 5 packages in 159ms                                         
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/202.88 KiB          
⠙ Preparing packages... (0/1)------------------- 14.90 KiB/202.88 KiB        
⠙ Preparing packages... (0/1)------------------- 30.90 KiB/202.88 KiB        
⠙ Preparing packages... (0/1)------------------- 46.90 KiB/202.88 KiB        
⠙ Preparing packages... (0/1)------------------- 62.90 KiB/202.88 KiB        
⠙ Preparing packages... (0/1)------------------- 78.90 KiB/202.88 KiB        
⠙ Preparing packages... (0/1)---------------- 94.90 KiB/202.88 KiB        
⠙ Preparing packages... (0/1)m-------------- 110.90 KiB/202.88 KiB       
⠙ Preparing packages... (0/1)30m------------ 126.90 KiB/202.88 KiB       
⠙ Preparing packages... (0/1)---------- 142.90 KiB/202.88 KiB       
⠙ Prepar

In [2]:
# Try to map GitHub usernames to countries with geopy
!uv pip install geopy pyyaml pandas tqdm

import time
import pandas as pd
import yaml
from tqdm import tqdm
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError


MANUAL_USER_MAPPING: dict[str, str] = {
    "benedictsaunders": "United Kingdom",  # Watford/Coventry
    "jonringer": "United States",  # Near Seattle, WA
}

# Load CSV
df = pd.read_csv("pr_info.csv")

# Geocoder Setup
geolocator = Nominatim(user_agent="geo_pr_mapper")


def safe_geocode(loc: str) -> str | None:
    try:
        location = geolocator.geocode(
            loc, addressdetails=True, language="en", timeout=10
        )
        if location and "country" in location.raw["address"]:
            return location.raw["address"]["country"]
    except (GeocoderTimedOut, GeocoderServiceError):
        pass

    print(f"geopy cannot resolve {username=} {loc=}")
    return None


# Map GitHub username to country
from_github: dict[str, str] = {}
unresolved: dict[str, str] = {}

for _, row in tqdm(df.iterrows(), total=len(df), desc="Geocoding users"):
    username = row["username"]
    location = row["location"]
    if pd.isna(location) or not location.strip():
        unresolved[username] = "NA"
        continue

    country = safe_geocode(location)

    if country:
        from_github[username] = country
    elif username in MANUAL_USER_MAPPING:
        from_github[username] = MANUAL_USER_MAPPING[username]
    else:
        print(f"Cannot resolve location for {username}")
        unresolved[username] = location

    time.sleep(1)  # rate limit

# Output YAML
with open("user_to_country.yaml", "w", encoding="utf-8") as f:
    output = {
        "from_github": from_github,
        "unresolved": unresolved,
    }
    yaml.dump(output, f, allow_unicode=True, sort_keys=False)

Using Python 3.14.3 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Audited 4 packages in 1ms


Geocoding users:  49%|████▉     | 135/276 [01:39<01:47,  1.32it/s]

geopy cannot resolve username='jonringer' loc='Near Seattle, WA'


Geocoding users: 100%|██████████| 276/276 [03:14<00:00,  1.42it/s]


In [ ]:
# Get info from pmg-doc-team page as secondary source
# https://pymatgen.org/team.html

!uv pip install pandas requests lxml

from io import StringIO

import requests
import pandas as pd

# Parse all tables (skip lead maintainers)
response = requests.get("https://pymatgen.org/team.html")
response.raise_for_status()
tables = pd.read_html(StringIO(response.text))[1:]

pmg_team_page_user_info: dict[str, str] = {}

for i, table in enumerate(tables):
    for _, row in table.iterrows():
        pmg_team_page_user_info[str(row["Github"]).strip()] = str(
            row["Position"]
        ).strip()

# print(pmg_team_page_user_info)

Using Python 3.14.3 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Audited 3 packages in 1ms


ModuleNotFoundError: No module named 'requets'

In [1]:
# Use LLM to map affiliation to countries in pymatgen team page
!uv pip install openai pycountry

import yaml
import pycountry
from openai import OpenAI


# Load existing YAML
with open("user_to_country.yaml", encoding="utf-8") as f:
    data = yaml.safe_load(f)

unresolved_users: list[str] = data.get("unresolved")

# Build target input: only unresolved users present in team page
affiliations_for_llm: dict[str, str] = {
    username: pmg_team_page_user_info[username]
    for username in unresolved_users
    if username in pmg_team_page_user_info
}

# Step 1: Construct LLM prompt
affiliation_lines = [
    f"{username}: {affiliation}"
    for username, affiliation in affiliations_for_llm.items()
]
prompt = f"""
    You are a helpful assistant. Given the following list of usernames and affiliations,
    please return a mapping of username to country (in ISO 3166-1 alpha-3 country format),
    where the country is where the affiliation is located. If not possible, please use NA as the country.

    Input:
        {"\n".join(affiliation_lines)}

    Output (as YAML, avoid markdown format):
    """

client = OpenAI()
chat_response = client.responses.create(
    model="gpt-4o",
    input=prompt,
)

yaml_output = chat_response.output_text[7:-3]  # drop markdown code fence

yaml_output = yaml.safe_load(yaml_output)


def iso3_to_full_name(code: str) -> str:
    try:
        return pycountry.countries.get(alpha_3=code.upper()).name
    except (AttributeError, KeyError):
        return "NA"


yaml_output = {
    str(username): iso3_to_full_name(code)
    for username, code in yaml_output.items()
    if code != "NA"
}

# Update user to country mapping YAML file
data.setdefault("from_pmg_doc", {})
data["from_pmg_doc"].update(yaml_output)

data["unresolved"] = {
    username: value
    for username, value in data["unresolved"].items()
    if username not in yaml_output.keys()
}

with open("user_to_country.yaml", "w", encoding="utf-8") as f:
    yaml.dump(data, f, allow_unicode=True, sort_keys=False)

print(
    "✅ user_to_country.yaml updated: added to 'from_pmg_doc' and removed from 'unresolved'."
)

Using Python 3.14.3 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Audited 2 packages in 1ms


NameError: name 'pmg_team_page_user_info' is not defined